# Lab 02: Tokens and Embeddings
**Course:** Large Language Models and Query Engineering
**Instructor:** Dr. Akram Fatayer
**Week 2 · Session B (lab)** · Supports CLO1 · Ends with **mini-assignment 1**

<a href="https://colab.research.google.com/github/akramfatayer/zuj-llm/blob/main/labs/lab-02-tokens-embeddings/Lab_02_Tokens_and_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## Plan for today (75 minutes)

| Time | What |
|---|---|
| 10 min | Brief: three tokenizer algorithms, and what "fertility" measures |
| 55 min | Parts A–C, in pairs |
| 10 min | Part D: check your results, read the mini-assignment brief |

By the end of this lab you will be able to:

1. Load several real tokenizers and see how BPE, WordPiece and SentencePiece split the same text.
2. Measure **fertility** (tokens per word) and turn the Arabic–English gap into cost and context length.
3. Show the difference between a **static** token embedding and a **contextual** one.
4. Use a multilingual embedding model to match Arabic sentences to their English translations.

Cells marked `# TODO` are yours to complete; the check cell after each prints ✅. Nothing here needs a GPU, so the lab also runs on a laptop.

## Part 0 — Setup (5 min)

Only the tokenizers (a few MB each) and one small embedding model (about 470 MB) are downloaded. On the lab machines they are already cached.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install -U "transformers>=4.45"

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore", message=".*sequence length is longer than.*")

# name shown in tables -> Hugging Face id; the algorithm each one uses is in the comment
TOKENIZERS = {
    "GPT-2": "openai-community/gpt2",                     # byte-level BPE, trained on English web text
    "Qwen2.5": "Qwen/Qwen2.5-1.5B-Instruct",              # byte-level BPE, multilingual (our lab model)
    "mBERT": "google-bert/bert-base-multilingual-cased",  # WordPiece, 104 languages
    "XLM-R": "FacebookAI/xlm-roberta-base",               # SentencePiece (unigram), 100 languages
    "AraBERT": "aubmindlab/bert-base-arabertv02",         # WordPiece, trained on Arabic
}
tokenizers = {name: AutoTokenizer.from_pretrained(hf_id) for name, hf_id in TOKENIZERS.items()}
for name, tok in tokenizers.items():
    print(f"{name:<8} vocabulary: {len(tok):>7,} tokens")

## Part A — One sentence, five tokenizers (10 min)

`pieces()` shows the tokens as the tokenizer stores them. Each algorithm marks word boundaries differently:

- **Byte-level BPE** (GPT-2, Qwen): a leading space is part of the token, e.g. `' students'`. Pieces of a multi-byte character can appear as `�`.
- **WordPiece** (mBERT, AraBERT): a token that continues a word starts with `##`.
- **SentencePiece** (XLM-R): a token that starts a word begins with `▁`.

**Before you run the next cell:** predict which tokenizer needs the most tokens for the Arabic sentence, and which the fewest. Write your prediction here: *…*

In [ ]:
def pieces(tok, text):
    """The tokens of `text` as readable strings, without special tokens such as [CLS]."""
    ids = tok(text, add_special_tokens=False).input_ids
    if tok.is_fast and tok.backend_tokenizer.pre_tokenizer is not None and "ByteLevel" in str(tok.backend_tokenizer.pre_tokenizer):
        return [tok.decode([i]) for i in ids]  # byte-level BPE: decode each id to see the text it covers
    return tok.convert_ids_to_tokens(ids)


english = "The students travel from Irbid to Aqaba."
arabic = "يسافر الطلاب من إربد إلى العقبة."
for name, tok in tokenizers.items():
    for text in (english, arabic):
        p = pieces(tok, text)
        print(f"{name:<8} {len(p):>3} tokens  {p}")
    print()

**Your answer:** was your prediction right? Why does GPT-2 need so many tokens for Arabic? (Hint: what text was its vocabulary learned from?)

*Write your answer here.*

## Part B — Fertility and cost (20 min)

One sentence proves nothing. Below are 20 sentence pairs (English and Modern Standard Arabic, same meaning) and five of them again in Jordanian Arabic.

**Fertility** = number of tokens ÷ number of words. A fertility of 1.0 means one token per word; higher means the tokenizer chops words into more pieces.

In [ ]:
pairs = [
    ("The university library opens at eight in the morning.", "تفتح مكتبة الجامعة في الثامنة صباحاً."),
    ("Students must register for their courses before the semester begins.", "يجب على الطلاب التسجيل في مقرراتهم قبل بداية الفصل الدراسي."),
    ("The final exam will be held in the main hall.", "سيُعقد الامتحان النهائي في القاعة الرئيسية."),
    ("Amman is the capital and the largest city of Jordan.", "عمّان هي عاصمة الأردن وأكبر مدنه."),
    ("The weather in Aqaba is warm throughout the year.", "الطقس في العقبة دافئ طوال العام."),
    ("Petra was carved into rose-coloured rock more than two thousand years ago.", "نُحتت البتراء في الصخر الوردي قبل أكثر من ألفي عام."),
    ("Please submit the assignment through the learning management system.", "يُرجى تسليم الواجب عبر نظام إدارة التعلّم."),
    ("The lab computers have powerful graphics processing units.", "تحتوي حواسيب المختبر على وحدات معالجة رسوميات قوية."),
    ("A language model predicts the next word in a sentence.", "يتنبأ النموذج اللغوي بالكلمة التالية في الجملة."),
    ("My brother works as an engineer in Irbid.", "يعمل أخي مهندساً في إربد."),
    ("We will meet at the café after the lecture.", "سنلتقي في المقهى بعد المحاضرة."),
    ("The Dead Sea is the lowest point on land.", "البحر الميت أخفض نقطة على اليابسة."),
    ("Olive trees grow well in the north of the country.", "تنمو أشجار الزيتون جيداً في شمال البلاد."),
    ("The bus to the university leaves every fifteen minutes.", "تغادر الحافلة إلى الجامعة كل خمس عشرة دقيقة."),
    ("Reading every day improves your vocabulary.", "تُحسّن القراءة اليومية حصيلتك من المفردات."),
    ("The team presented its project to the committee on Sunday.", "عرض الفريق مشروعه على اللجنة يوم الأحد."),
    ("Water is scarce, so we must use it carefully.", "الماء شحيح، لذا يجب أن نستخدمه بحذر."),
    ("The doctor advised him to rest for three days.", "نصحه الطبيب بالراحة ثلاثة أيام."),
    ("Artificial intelligence is changing how we work and learn.", "يغيّر الذكاء الاصطناعي طريقة عملنا وتعلّمنا."),
    ("Thank you for your help with the experiment.", "شكراً لمساعدتك في التجربة."),
]
# Jordanian Arabic versions of pairs 1, 7, 11, 17 and 20
dialect = [
    "مكتبة الجامعة بتفتح الساعة ثمانية الصبح.",
    "لو سمحت سلّم الواجب على المودل.",
    "رح نلتقي بالكافيه بعد المحاضرة.",
    "المي قليلة، فلازم نستعملها بحرص.",
    "يسلموا إيديك على مساعدتك بالتجربة.",
]
english_sents = [en for en, _ in pairs]
msa_sents = [ar for _, ar in pairs]
print(len(pairs), "pairs,", len(dialect), "dialect sentences")

### Exercise B1

Complete `fertility()`. Count tokens **without** special tokens (pass `add_special_tokens=False`), and count words by splitting on spaces.

In [ ]:
def fertility(tok, sentences):
    """Total tokens divided by total words over a list of sentences."""
    # TODO: count the tokens of every sentence (no special tokens)
    n_tokens = None
    # TODO: count the words of every sentence
    n_words = None
    return n_tokens / n_words

In [ ]:
# Check
assert abs(fertility(tokenizers["GPT-2"], ["hello world"]) - 1.0) < 1e-9, "'hello world' is 2 GPT-2 tokens and 2 words"
print("✅ fertility() works")

rows = []
for name, tok in tokenizers.items():
    f_en, f_msa, f_dia = (fertility(tok, s) for s in (english_sents, msa_sents, dialect))
    en_tokens = sum(len(tok(s, add_special_tokens=False).input_ids) for s in english_sents)
    ar_tokens = sum(len(tok(s, add_special_tokens=False).input_ids) for s in msa_sents)
    rows.append({"tokenizer": name, "English": f_en, "MSA": f_msa, "Dialect": f_dia, "Arabic ÷ English tokens": ar_tokens / en_tokens})
results = pd.DataFrame(rows).set_index("tokenizer").round(2)
display(results)

ax = results[["English", "MSA", "Dialect"]].plot.bar(figsize=(8, 3.5), rot=0, color=["#14213D", "#A84E14", "#F0A35E"])
ax.set_ylabel("tokens per word")
ax.set_title("Fertility on the same 20 sentences")
plt.tight_layout()
plt.show()

Note that "Arabic ÷ English tokens" compares the *same meaning*. Fertility alone can mislead, because Arabic attaches words English writes separately: "سنلتقي" is one word, "we will meet" is three.

### Exercise B2 — What the gap costs

Open-weight models in our lab cost GPU time, not money per token, but the arithmetic is the same: every token is one more step of work, and the context window is measured in tokens. Complete the cell: for each tokenizer, how many **words** of English and of Arabic fit in a 4,096-token context window?

In [ ]:
CONTEXT = 4096
for name in results.index:
    # TODO: words that fit = CONTEXT ÷ fertility
    en_words = None
    ar_words = None
    print(f"{name:<8} English: {en_words:>6,.0f} words   Arabic (MSA): {ar_words:>6,.0f} words")

**Questions** (answer in two or three sentences each):

1. Which tokenizer treats Arabic and English most equally? Which tokenizer would you avoid for an Arabic chatbot, and what would it cost you?
2. Is dialect more expensive than MSA? Why might that be, given how the vocabularies were learned?

*Write your answers here.*

## Part C — Embeddings: static and contextual (20 min)

A model first turns each token ID into a vector by looking it up in an **embedding table**. That vector is *static*: the same token always gets the same vector. The layers above then mix in the surrounding words, producing a *contextual* vector for each position.

We use a small multilingual sentence-embedding model (12 layers, trained so that a sentence and its translation get similar vectors).

In [ ]:
EMB_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
emb_tok = AutoTokenizer.from_pretrained(EMB_ID)
emb_model = AutoModel.from_pretrained(EMB_ID).eval()
table = emb_model.get_input_embeddings().weight  # the static embedding table: one row per token
print(f"Embedding table: {table.shape[0]:,} tokens × {table.shape[1]} dimensions")


def cosine(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=-1).item()

### C1 — Same word, different meaning

"bank" means two things in English, and "العين" means both *the eye* and *the spring (of water)* in Arabic. The static vector cannot tell them apart. Does the contextual vector?

In [ ]:
@torch.no_grad()
def word_vectors(sentence, word):
    """Static and contextual vectors of `word` in `sentence` (averaged over its tokens)."""
    enc = emb_tok(sentence, return_tensors="pt", return_offsets_mapping=True)
    start = sentence.index(word)
    end = start + len(word)
    offsets = enc.pop("offset_mapping")[0]
    positions = [i for i, (s, e) in enumerate(offsets.tolist()) if e > start and s < end and e > s]
    hidden = emb_model(**enc).last_hidden_state[0]
    static = table[enc.input_ids[0, positions]].mean(0)
    contextual = hidden[positions].mean(0)
    return static, contextual


def compare(word, sentences):
    vecs = [word_vectors(s, word) for s in sentences]
    print(f"{word!r}")
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            print(f"  {i + 1} vs {j + 1}:  static {cosine(vecs[i][0], vecs[j][0]):.2f}   contextual {cosine(vecs[i][1], vecs[j][1]):.2f}")
    for i, s in enumerate(sentences):
        print(f"  {i + 1}. {s}")


compare("bank", [
    "We sat on the bank of the river and watched the water.",
    "She deposited her salary at the bank this morning.",
    "The bank approved my loan after a week.",
])
compare("العين", [
    "شربنا الماء البارد من العين في أسفل الوادي.",
    "أصيب اللاعب في العين خلال المباراة.",
    "فحص الطبيب العين ووصف قطرة للمريض.",
])

**Question:** which pair is the most similar in context, and why are all the static similarities exactly 1.00?

*Write your answer here.*

### C2 — Matching translations

The model was trained so that a sentence and its translation land close together. If that worked, each Arabic sentence's nearest English sentence should be its own translation. Complete the TODO to find each Arabic sentence's best match.

In [ ]:
@torch.no_grad()
def embed(sentences):
    """One unit-length vector per sentence: the mean of its contextual token vectors."""
    enc = emb_tok(sentences, padding=True, truncation=True, return_tensors="pt")
    hidden = emb_model(**enc).last_hidden_state
    mask = enc.attention_mask.unsqueeze(-1).float()
    mean = (hidden * mask).sum(1) / mask.sum(1)
    return torch.nn.functional.normalize(mean, dim=-1)


E = embed(english_sents)
A = embed(msa_sents)
similarity = A @ E.T  # row i, column j: Arabic sentence i vs English sentence j
print("similarity matrix:", tuple(similarity.shape))

In [ ]:
# TODO: for each Arabic sentence (row), the index of the most similar English sentence (hint: argmax over dim=1)
best = None
correct = None  # TODO: how many rows found their own translation (best[i] == i)?
print(f"Top-1 accuracy: {correct}/{len(pairs)}")

In [ ]:
# Check, and show any misses
assert best is not None and len(best) == len(pairs), "best should have one entry per Arabic sentence"
print("✅ matching works")
for i, j in enumerate(best.tolist()):
    if i != j:
        print(f"MISS  {msa_sents[i]}\n   -> {english_sents[j]}  (similarity {similarity[i, j]:.2f})")

plt.figure(figsize=(5, 4.5))
plt.imshow(similarity.numpy(), cmap="Oranges", vmin=0, vmax=1)
plt.colorbar(label="cosine similarity")
plt.xlabel("English sentence")
plt.ylabel("Arabic sentence")
plt.title("A bright diagonal = translations found")
plt.tight_layout()
plt.show()

**Question:** look at the dialect sentences. Do they still land closest to the right English sentence? Run the cell below and explain what you see.

In [ ]:
D = embed(dialect)
for k, (sentence, idx) in enumerate(zip(dialect, [0, 6, 10, 16, 19])):
    scores = (D[k] @ E.T)
    top = int(scores.argmax())
    print(f"{'ok  ' if top == idx else 'MISS'} {sentence}\n     -> {english_sents[top]}  ({scores[top]:.2f}; own translation {scores[idx]:.2f})")

## Part D — Check and reflect (10 min)

1. **One sentence each:** what is a token, and what is an embedding?
2. A classmate says "Arabic is simply a harder language for computers." Using your fertility table, give a more precise explanation.
3. Why can a model match "رح نلتقي بالكافيه" to "We will meet at the café" although the words share no letters?

*Write your answers here.*

---

## Mini-assignment 1: Tokenizer audit

Individual, due at the start of the week 3 lab, 3% of the course grade. Read the brief: [MA1_Tokenizer_Audit.md](https://github.com/akramfatayer/zuj-llm/blob/main/labs/lab-02-tokens-embeddings/MA1_Tokenizer_Audit.md). You will extend this notebook, so save a copy now (*File → Save a copy in Drive* on Colab).

**Before next week:** finish sections 4–6 of the interactive module [From Counting to Tokens](https://akramfatayer.github.io/courses/llm/learn/1#s4) and paste its completion code into Moodle. Read *Hands-On Large Language Models*, chapter 3, and the attention section of Vaswani et al. (2017). Quiz 1 (weeks 1–2) is at the start of next week's lecture.